# Gemma 4 12B prompting — CEFR classification (test)

**Constrained label scoring instead of free generation.** The model is run
   in-process via Unsloth rather than through LM Studio, and the 8 candidate
   labels are scored by likelihood. This yields a probability distribution
   (so ECE and confidence bands are computable, as for the encoders)

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import json, random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from metrics import evaluate_predictions
from prompts import LEVELS, SYSTEM_PROMPT, CLASSIFICATION_REQUEST

LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}
NUM_CLASSES = len(LEVELS)

SPLIT = "test"


@dataclass
class CFG:
    model_id: str = "unsloth/gemma-4-12b-it"     # base model, no adapter
    data_dir: str = "data"
    text_col: str = "text"
    label_col: str = "label"
    max_seq_length: int = 8192                   # 24-shot prompts are long
    load_in_4bit: bool = True
    chunk: int = 2                               # candidates per forward pass
    empty_cache_every: int = 10
    run_dir: str = "runs/gemma4_12b_prompting"


cfg = CFG()
os.makedirs(cfg.run_dir, exist_ok=True)

## Data

In [3]:
df_train = pd.read_json(f"{cfg.data_dir}/train.jsonl", lines=True)
df = pd.read_json(f"{cfg.data_dir}/{SPLIT}.jsonl", lines=True)

df_train = df_train[df_train[cfg.label_col].isin(LEVELS)].reset_index(drop=True)
df = df[df[cfg.label_col].isin(LEVELS)].reset_index(drop=True)
print(f"train {len(df_train)} | {SPLIT} {len(df)}")

train 3797 | test 604


## Few-shot example pools

Unchanged from the poster: balanced and topic-diverse, with 8 and 16 as true
subsets of 24, then shuffled independently for prompt order.

In [4]:
# def sample_few_shot_examples(df, n_per_class, levels=LEVELS, random_state=42,
#                              topic_col="topic"):
#     """Balanced, topic-diverse examples. Level-grouped, not shuffled."""
#     examples = []
#     rng = random.Random(random_state)

#     for level in levels:
#         subset = df[df["label"] == level]
#         n = min(n_per_class, len(subset))

#         if topic_col in subset.columns and subset[topic_col].nunique() >= n:
#             available_topics = subset[topic_col].unique().tolist()
#             rng.shuffle(available_topics)
#             sampled = pd.concat([
#                 subset[subset[topic_col] == t].sample(n=1, random_state=random_state)
#                 for t in available_topics[:n]
#             ])
#         else:
#             sampled = subset.sample(n=n, random_state=random_state)

#         for _, row in sampled.iterrows():
#             examples.append({"text": row["text"], "label": level,
#                              "topic": row.get(topic_col, None)})
#     return examples


# def take_first_n_per_level(grouped, n_per_class, levels=LEVELS):
#     out = []
#     for level in levels:
#         out.extend([ex for ex in grouped if ex["label"] == level][:n_per_class])
#     return out


# grouped = sample_few_shot_examples(df_train, n_per_class=3)

# rng = random.Random(42)
# EXAMPLE_SETS = {}
# for n_shots, per_level in [(8, 1), (16, 2), (24, 3)]:
#     s = take_first_n_per_level(grouped, per_level)
#     rng.shuffle(s)
#     EXAMPLE_SETS[n_shots] = s
#     print(f"{n_shots}-shot: {len(s)} examples")

## Model

Base model, no adapter. Everything else matches the fine-tuned notebook.

In [5]:
# from unsloth import FastLanguageModel

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=cfg.model_id,
#     max_seq_length=cfg.max_seq_length,
#     dtype=None,
#     load_in_4bit=cfg.load_in_4bit,
# )
# FastLanguageModel.for_inference(model)
# tok = getattr(tokenizer, "tokenizer", tokenizer)


# def vram():
#     return (torch.cuda.memory_allocated() / 1e9,
#             torch.cuda.memory_reserved() / 1e9)


# print("allocated %.1f GB | reserved %.1f GB" % vram())

## Prompt construction

Few-shot examples are prior user/assistant turns, the assistant turn being the
bare CEFR level. Identical in form to the fine-tuning targets.

In [6]:
# def build_prompt_messages(text, shots=()):
#     msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
#     for ex in shots:
#         msgs.append({"role": "user",
#                      "content": CLASSIFICATION_REQUEST + str(ex["text"])})
#         msgs.append({"role": "assistant", "content": ex["label"]})
#     msgs.append({"role": "user", "content": CLASSIFICATION_REQUEST + str(text)})
#     return msgs


# # Check prompt length for the longest condition before running anything.
# _p = tok.apply_chat_template(
#     build_prompt_messages(df[cfg.text_col].iloc[0], EXAMPLE_SETS[24]),
#     tokenize=True, add_generation_prompt=True)
# print(f"24-shot prompt: {len(_p)} tokens (max_seq_length {cfg.max_seq_length})")
# print(repr(tok.decode(_p[-60:])))

## Constrained label scoring

The end-of-turn token is appended to each candidate. Without it the bare
labels are prefixes of one another (A2 = [236776, 236778],
A2+ = [236776, 236778, 236862]), so summed log-probs made every plus level
strictly lower than its base level and they could never be predicted.

Candidates are scored `cfg.chunk` at a time. Long few-shot prompts times 8
candidates in one batch overflows VRAM.

In [7]:
# EOT_ID = tok.convert_tokens_to_ids("<turn|>")
# assert EOT_ID is not None and EOT_ID >= 0, "check the eot token string"

# LABEL_TOKEN_IDS = {lvl: tok(lvl, add_special_tokens=False).input_ids + [EOT_ID]
#                    for lvl in LEVELS}
# LAB_IDS = [LABEL_TOKEN_IDS[lvl] for lvl in LEVELS]
# MAX_LAB = max(len(l) for l in LAB_IDS)
# print({k: len(v) for k, v in LABEL_TOKEN_IDS.items()})


# @torch.no_grad()
# def score_labels(text, shots=(), chunk=None):
#     """Return (8,) softmax distribution over LEVELS."""
#     chunk = chunk or cfg.chunk
#     prompt_ids = tok.apply_chat_template(
#         build_prompt_messages(text, shots), tokenize=True,
#         add_generation_prompt=True)
#     start = len(prompt_ids)
#     pad_id = tok.pad_token_id
#     ll = np.zeros(NUM_CLASSES)

#     for lo in range(0, NUM_CLASSES, chunk):
#         idx = list(range(lo, min(lo + chunk, NUM_CLASSES)))
#         seqs = [prompt_ids + LAB_IDS[i] for i in idx]
#         maxlen = max(len(s) for s in seqs)

#         input_ids = torch.full((len(seqs), maxlen), pad_id, dtype=torch.long)
#         attn = torch.zeros((len(seqs), maxlen), dtype=torch.long)
#         for r, s in enumerate(seqs):
#             input_ids[r, :len(s)] = torch.tensor(s)
#             attn[r, :len(s)] = 1

#         input_ids, attn = input_ids.to(model.device), attn.to(model.device)
#         logits = model(input_ids=input_ids, attention_mask=attn).logits
#         sliced = logits[:, start - 1: start - 1 + MAX_LAB, :]
#         logprobs = torch.log_softmax(sliced.float(), dim=-1)

#         for r, i in enumerate(idx):
#             ll[i] = sum(logprobs[r, j, input_ids[r, start + j]].item()
#                         for j in range(len(LAB_IDS[i])))
#         del logits, sliced, logprobs

#     ll -= ll.max()
#     e = np.exp(ll)
#     return e / e.sum()

## Run in small batches

In [10]:
# %%
import subprocess, sys

CONDITIONS = ["zero_shot", "few_shot_8", "few_shot_16", "few_shot_24"]
BOUNDS = [(0,75), (75,150), (150,225), (225,300),
          (300,375), (375,450), (450,525), (525,604)]
TIMEOUT = 2700          # 45 min per chunk, then kill and move on

failed = []
for cond in CONDITIONS:
    for a, b in BOUNDS:
        print(f"\n=== {cond} {a}-{b} ===", flush=True)
        p = subprocess.Popen(
            [sys.executable, "-u", "score_chunk.py", cond, str(a), str(b), "test"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=0, encoding="utf-8", errors="replace")
        try:
            while True:
                c = p.stdout.read(1)
                if not c:
                    break
                print(c, end="", flush=True)
            if p.wait(timeout=60) != 0:
                failed.append((cond, a, b))
        except subprocess.TimeoutExpired:
            p.kill()
            failed.append((cond, a, b))
            print(f"\nTIMEOUT: {cond} {a}-{b}", flush=True)

print("\nfailed:", failed)


=== zero_shot 0-75 ===
exists, skipping: runs/gemma4_12b_prompting/test_zero_shot_0_75.npy

=== zero_shot 75-150 ===
exists, skipping: runs/gemma4_12b_prompting/test_zero_shot_75_150.npy

=== zero_shot 150-225 ===
exists, skipping: runs/gemma4_12b_prompting/test_zero_shot_150_225.npy

=== zero_shot 225-300 ===
exists, skipping: runs/gemma4_12b_prompting/test_zero_shot_225_300.npy

=== zero_shot 300-375 ===
exists, skipping: runs/gemma4_12b_prompting/test_zero_shot_300_375.npy

=== zero_shot 375-450 ===
exists, skipping: runs/gemma4_12b_prompting/test_zero_shot_375_450.npy

=== zero_shot 450-525 ===
exists, skipping: runs/gemma4_12b_prompting/test_zero_shot_450_525.npy

=== zero_shot 525-604 ===
exists, skipping: runs/gemma4_12b_prompting/test_zero_shot_525_604.npy

=== few_shot_8 0-75 ===
exists, skipping: runs/gemma4_12b_prompting/test_few_shot_8_0_75.npy

=== few_shot_8 75-150 ===
exists, skipping: runs/gemma4_12b_prompting/test_few_shot_8_75_150.npy

=== few_shot_8 150-225 ===
exis

## Collect conditions

Run once every condition's `.npy` files exist. Chunked runs are concatenated
in order.

In [11]:
import glob

CONDITIONS = ["zero_shot", "few_shot_8", "few_shot_16", "few_shot_24"]
gold = df[cfg.label_col].map(LABEL2ID).to_numpy()

results, per_condition = [], {}
for cond in CONDITIONS:
    files = sorted(glob.glob(f"{cfg.run_dir}/{SPLIT}_{cond}_*.npy"),
                   key=lambda f: int(f.rsplit("_", 2)[1]))
    if not files:
        print(f"{cond}: no files yet")
        continue

    p = np.vstack([np.load(f) for f in files])
    assert len(p) == len(gold), f"{cond}: {len(p)} rows vs {len(gold)} gold"
    per_condition[cond] = p

    pred = p.argmax(axis=1)
    m = evaluate_predictions(gold, pred)
    m["condition"] = cond
    m["mean_p_pred"] = float(p[np.arange(len(pred)), pred].mean())
    results.append(m)

res_df = pd.DataFrame(results).set_index("condition")
print(res_df.round(3).to_string())
res_df.to_csv(f"{cfg.run_dir}/{SPLIT}_metrics.csv")

               qwk    mae  accuracy  adjacent_accuracy  precision_macro  recall_macro  f1_macro  precision_weighted  recall_weighted  f1_weighted  mean_p_pred
condition                                                                                                                                                     
zero_shot    0.693  1.002     0.290              0.755            0.264         0.268     0.184               0.314            0.290        0.224        0.894
few_shot_8   0.759  0.810     0.333              0.871            0.284         0.276     0.243               0.342            0.333        0.277        0.868
few_shot_16  0.782  0.808     0.334              0.873            0.265         0.307     0.245               0.324            0.334        0.277        0.851
few_shot_24  0.832  0.700     0.401              0.904            0.409         0.367     0.365               0.427            0.401        0.388        0.829


In [12]:
# %%
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

COND = "few_shot_24"

p = per_condition[COND]
pred = p.argmax(axis=1)

counts = confusion_matrix(gold, pred, labels=list(range(NUM_CLASSES)))
row_sums = counts.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    norm = np.nan_to_num(np.divide(counts, row_sums, where=row_sums != 0))

annot = np.empty_like(counts, dtype=object)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        annot[i, j] = (f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"
                       if counts[i, j] else "")

fig = go.Figure(go.Heatmap(
    z=norm, x=LEVELS, y=LEVELS,
    text=annot, texttemplate="%{text}",
    hoverongaps=False,
    colorscale="Greys", zmin=0, zmax=1,
    colorbar=dict(title="Row-normalized"),
))
fig.update_xaxes(title_text="Predicted")
fig.update_yaxes(title_text="True", autorange="reversed")
fig.update_layout(
    title=dict(text=f"Gemma 4 12B prompted, {COND} — {SPLIT}"),
    font=dict(family="Arial", size=16, color="black"),
    width=620, height=560,
    margin=dict(l=80, r=100, t=100, b=80),
)
fig.show()

## Model Confidence

In [13]:
# %%
COND = "few_shot_24"

p = per_condition[COND]
pred = p.argmax(axis=1)
d = pd.DataFrame({
    "correct": (gold == pred).astype(int),
    "adjacent": (np.abs(gold - pred) <= 1).astype(int),
    "p_pred": p[np.arange(len(pred)), pred],
})

BANDS = [(0.90, 1.01, "> .90"), (0.80, 0.90, ".80–.90"),
         (0.70, 0.80, ".70–.80"), (0.60, 0.70, ".60–.70"), (0.00, 0.60, "≤ .60")]

n = len(d)
rows = []
for lo, hi, name in BANDS:
    s = d[(d["p_pred"] >= lo) & (d["p_pred"] < hi)]
    rows.append(dict(
        conf=name, preds=len(s), share=f"{len(s)/n*100:.0f}%",
        acc=round(s["correct"].mean(), 2) if len(s) else None,
        adj=round(s["adjacent"].mean(), 2) if len(s) else None,
    ))

print(f"{COND}  mean p_pred {d['p_pred'].mean():.3f}")
print(pd.DataFrame(rows).to_string(index=False))


def ece(d, n_bins=10):
    b = pd.cut(d["p_pred"], np.linspace(0, 1, n_bins + 1))
    g = d.groupby(b, observed=True)
    gap = (g["correct"].mean() - g["p_pred"].mean()).abs()
    return float((gap * g.size()).sum() / len(d))


print(f"ECE {ece(d):.4f}")

few_shot_24  mean p_pred 0.829
   conf  preds share  acc  adj
  > .90    282   47% 0.42 0.92
.80–.90    102   17% 0.38 0.85
.70–.80     74   12% 0.38 0.92
.60–.70     71   12% 0.39 0.86
  ≤ .60     75   12% 0.37 0.93
ECE 0.4280


## Per-condition prediction files

Same columns as the encoder and fine-tuned runs, so the calibration and AUROC
cells read them without modification.

In [14]:
for cond, p in per_condition.items():
    pred = p.argmax(axis=1)
    srt = np.sort(p, axis=1)[:, ::-1]
    rows = []
    for i, (g, q) in enumerate(zip(gold, pred)):
        g, q = int(g), int(q)
        lo, hi = max(0, q - 1), min(NUM_CLASSES - 1, q + 1)
        rows.append(dict(
            gold=g, pred=q, gold_label=LEVELS[g], pred_label=LEVELS[q],
            correct=int(g == q), adjacent=int(abs(g - q) <= 1),
            p_pred=float(p[i, q]),
            p_adjacent=float(p[i, lo:hi + 1].sum()),
            margin=float(srt[i, 0] - srt[i, 1]),
            entropy=float(-(p[i] * np.log(p[i] + 1e-12)).sum()),
            exp_level=float((p[i] * np.arange(NUM_CLASSES)).sum()),
            **{f"p_{lvl}": float(p[i, j]) for j, lvl in enumerate(LEVELS)},
        ))
    path = f"{cfg.run_dir}/{SPLIT}_{cond}_predictions.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    print(path)

runs/gemma4_12b_prompting/test_zero_shot_predictions.csv
runs/gemma4_12b_prompting/test_few_shot_8_predictions.csv
runs/gemma4_12b_prompting/test_few_shot_16_predictions.csv
runs/gemma4_12b_prompting/test_few_shot_24_predictions.csv
